# APIM ❤️ AI Foundry

## AI Foundry Toolbox lab
![flow](../../images/ai-foundry-toolbox.gif)

A [Foundry Toolbox](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/toolbox) is a managed resource in Azure AI Foundry that bundles tools—MCP servers, Web Search, Azure AI Search, Code Interpreter, and more—and exposes them through a single MCP-compatible endpoint:
`{project_endpoint}/toolboxes/{name}/mcp?api-version=v1`

This lab deploys Azure API Management in front of that endpoint. APIM accepts subscription key auth from clients and transparently injects a managed-identity Entra token when forwarding requests to the Toolbox, so consumers never need to handle Foundry credentials directly. APIM also adds rate limiting, per-subscription monitoring, and policy-based governance over every tool call.

**What this lab covers:**
- Deploy APIM and Azure AI Foundry with Bicep
- Deploy the vet-clinic and pet-insurance MCP servers as Azure Functions Flex Consumption apps
- Create a Foundry Toolbox that wraps both MCP servers
- Configure APIM to proxy the Toolbox MCP endpoint (subscription key → managed-identity Entra token)
- Verify tool discovery through the APIM proxy
- Run an OpenAI chat completion with Toolbox tools routed through APIM

### Prerequisites

- [Python 3.12 or later](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [uv](https://docs.astral.sh/uv/) — run `uv sync` from the repo root to install dependencies
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [signed in to your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription ID.
- Adjust the location parameters according to your preferences and [product availability by Azure region](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management).
- Adjust the models and versions according to [availability by region](https://learn.microsoft.com/azure/ai-services/openai/concepts/models).

In [ ]:
# Make sure you have 2.3.0 installed
%uv pip install 'azure-ai-projects==2.3.0'

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"  # change the name to match your naming style
resource_group_location = "swedencentral"  # change the location to match your requirements

# AI Services configuration
aiservices_config = [{"name": "foundry1", "location": "swedencentral"}]  # change the location to match your requirements

# Models configuration
models_config = [{"name": "gpt-5-mini", "publisher": "OpenAI", "version": "2025-08-07", "sku": "GlobalStandard", "capacity": 20}]

# APIM configuration
apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# Inference API configuration
inference_api_path = "inference"
inference_api_type = "AzureOpenAI"
inference_api_version = "2025-03-01-preview"
foundry_project_name = deployment_name

# Foundry Toolbox configuration
toolbox_name = "pet-care-toolbox"  # shared Toolbox that will contain both MCP servers
toolbox_mcp_path = "toolbox/mcp-native"  # APIM native MCP path

# MCP server source folders
vet_clinic_src = "mcp-servers/vet-clinic"
pet_insurance_src = "mcp-servers/pet-insurance"

utils.print_ok('Notebook initialized' )

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources deployed in the specified resource group. The template provisions:
- Azure API Management (inference API + Toolbox MCP proxy API)
- Azure AI Foundry (with gpt-5-mini)
- Azure Storage Account + Flex Consumption App Service Plan + two Function Apps (vet and pet insurance MCP servers)
- Log Analytics Workspace and Application Insights

In [ ]:
# Create the resource group if it doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name },
        "toolboxName": { "value": toolbox_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Get the deployment outputs

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", '"'))
    for subscription in apim_subscriptions:
        utils.print_info(f"Subscription Name: {subscription['name']}")
        utils.print_info(f"Subscription Key: ****{subscription['key'][-4:]}")
    api_key = apim_subscriptions[0].get("key")
    foundry_project_endpoint = utils.get_deployment_output(output, 'foundryProjectEndpoint', 'Foundry Project Endpoint')
    vet_function_app_name = utils.get_deployment_output(output, 'functionAppName', 'Vet Function App Name')
    vet_function_app_url = utils.get_deployment_output(output, 'functionAppUrl', 'Vet Function App URL')
    pet_insurance_function_app_name = utils.get_deployment_output(output, 'petInsuranceFunctionAppName', 'Pet Insurance Function App Name')
    pet_insurance_function_app_url = utils.get_deployment_output(output, 'petInsuranceFunctionAppUrl', 'Pet Insurance Function App URL')

# Derive the Toolbox MCP endpoint (used in subsequent steps)
toolbox_mcp_endpoint = f"{foundry_project_endpoint}/toolboxes/{toolbox_name}/mcp?api-version=v1"
apim_toolbox_mcp_url = f"{apim_resource_gateway_url}/{toolbox_mcp_path}"
utils.print_info(f"Foundry Toolbox MCP endpoint: {toolbox_mcp_endpoint}")
utils.print_info(f"APIM Toolbox MCP proxy URL: {apim_toolbox_mcp_url}")

<a id='4'></a>
### 4️⃣ Deploy the vet and pet insurance MCP servers to Azure Functions (Flex Consumption)

The [vet-clinic](mcp-servers/vet-clinic/function_app.py) and [pet-insurance](mcp-servers/pet-insurance/function_app.py) are Azure Functions apps that expose MCP tools with the `@app.mcp_tool()` decorator.

Because they are standard Azure Functions apps, deployment is a simple zip-deploy for each app - **no container or Docker build required**.

In [ ]:
import os
import zipfile

ignore_names = {".git", "__pycache__", ".venv", ".python_packages", "local.settings.json"}


def create_zip(source_dir, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(source_dir):
            dirs[:] = [d for d in dirs if d not in ignore_names and not d.startswith('.')]
            for file in files:
                if file.endswith('.zip'):
                    continue
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, source_dir)
                zf.write(file_path, arcname)


vet_zip_path = f"{vet_clinic_src}/vet-clinic.zip"
pet_insurance_zip_path = f"{pet_insurance_src}/pet-insurance.zip"

create_zip(vet_clinic_src, vet_zip_path)
create_zip(pet_insurance_src, pet_insurance_zip_path)

utils.print_ok(f"Created deployment zips: {vet_zip_path}, {pet_insurance_zip_path}")

# Deploy both zip packages to their Flex Consumption Function Apps - no container or Docker build required
utils.run(
    f"az functionapp deployment source config-zip -g {resource_group_name} -n {vet_function_app_name} --src {vet_zip_path}",
    f"Vet Clinic deployed to Function App '{vet_function_app_name}'",
    f"Failed to deploy vet-clinic to Function App")

utils.run(
    f"az functionapp deployment source config-zip -g {resource_group_name} -n {pet_insurance_function_app_name} --src {pet_insurance_zip_path}",
    f"Pet Insurance deployed to Function App '{pet_insurance_function_app_name}'",
    f"Failed to deploy pet-insurance to Function App")

<a id='5'></a>
### 5️⃣ Create the Foundry Toolbox

Use the [azure-ai-projects SDK](https://learn.microsoft.com/azure/ai-studio/how-to/develop/sdk-overview) to create a [Foundry Toolbox](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/toolbox) version with two [MCP tools](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/model-context-protocol) pointing to the vet-clinic and pet-insurance Function Apps.

The Azure Functions MCP extension exposes tools at:
`{vet_function_app_url}/runtime/webhooks/mcp`
`{pet_insurance_function_app_url}/runtime/webhooks/mcp`

The Toolbox wraps both endpoints and exposes everything through a single governed MCP endpoint:
`{foundry_project_endpoint}/toolboxes/{toolbox_name}/mcp?api-version=v1`

APIM is already pre-configured in Bicep to proxy this endpoint, so any call to `{apim_gateway_url}/toolbox/mcp-native` is transparently forwarded to the Toolbox with a managed-identity Entra token.

In [ ]:
# Azure Functions MCP extension endpoints
vet_toolbox_mcp_server_url = f"{vet_function_app_url}/runtime/webhooks/mcp"
pet_insurance_toolbox_mcp_server_url = f"{pet_insurance_function_app_url}/runtime/webhooks/mcp"

utils.print_info(f"Vet Toolbox MCP server URL: {vet_toolbox_mcp_server_url}")
utils.print_info(f"Pet Insurance Toolbox MCP server URL: {pet_insurance_toolbox_mcp_server_url}")

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool

project = AIProjectClient(endpoint=foundry_project_endpoint, credential=DefaultAzureCredential())

vet_toolbox_mcp_server_url = f"{vet_function_app_url}/runtime/webhooks/mcp"
pet_insurance_toolbox_mcp_server_url = f"{pet_insurance_function_app_url}/runtime/webhooks/mcp"

utils.print_info(f"Vet Toolbox MCP server URL: {vet_toolbox_mcp_server_url}")
utils.print_info(f"Pet Insurance Toolbox MCP server URL: {pet_insurance_toolbox_mcp_server_url}")

toolbox_version = project.toolboxes.create_version(
    name=toolbox_name,
    description="Pet care toolbox -- veterinary and insurance tools via Azure Functions MCP",
    tools=[
        MCPTool(server_label="vet", server_url=vet_toolbox_mcp_server_url, require_approval="never"),
        MCPTool(server_label="insurance", server_url=pet_insurance_toolbox_mcp_server_url, require_approval="never"),
    ]
)

utils.print_ok(f"Created Toolbox '{toolbox_name}' version: {toolbox_version.version}")
utils.print_info(f"Toolbox MCP endpoint: {toolbox_mcp_endpoint}")

<a id='6'></a>
### 🧪 Verify Toolbox tool discovery (direct — Entra token)

Connect directly to the Foundry Toolbox MCP endpoint using your DefaultAzureCredential to confirm the veterinary and insurance tools are available.

Tip: Use the [MCP Inspector](https://modelcontextprotocol.io/docs/tools/inspector) for interactive testing:
1. Run `npx @modelcontextprotocol/inspector` in a terminal
2. Set transport to **Streamable HTTP** and paste the Toolbox MCP endpoint

In [ ]:
# One-time notebook dependency setup for MCP tests
%pip install mcp nest_asyncio

In [ ]:
import asyncio
import httpx
import traceback
from azure.identity import DefaultAzureCredential
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
import nest_asyncio
nest_asyncio.apply()

async def list_toolbox_tools(endpoint, bearer_token):
    if "api-version=" not in endpoint:
        endpoint = f"{endpoint}{'&' if '?' in endpoint else '?'}api-version=v1"

    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Foundry-Features": "Toolboxes=V1Preview"
    }
    try:
        async with httpx.AsyncClient(headers=headers, timeout=60.0, follow_redirects=True) as mcp_http_client:
            async with streamable_http_client(endpoint, http_client=mcp_http_client) as streams:
                async with ClientSession(streams[0], streams[1]) as session:
                    await session.initialize()
                    response = await session.list_tools()
        print(f"✅ Connected to: {endpoint}")
        print("⚙️ Available tools:")
        for tool in response.tools:
            print(f"   - {tool.name}: {tool.description}")
    except Exception as e:
        print(f"❌ Error type: {type(e).__name__}")
        print(f"❌ Error: {e}")
        if hasattr(e, "exceptions"):
            for index, sub_error in enumerate(e.exceptions, start=1):
                print(f"   ↳ Sub-exception {index}: {type(sub_error).__name__}: {sub_error}")
        print("\n--- Traceback ---")
        print(traceback.format_exc())

# Get Entra token for Foundry (scope: https://ai.azure.com/.default)
token = DefaultAzureCredential().get_token("https://ai.azure.com/.default").token

# Test directly via the Foundry Toolbox endpoint
asyncio.run(list_toolbox_tools(toolbox_mcp_endpoint, token))

<a id='7'></a>
### 🧪 Verify Toolbox tool discovery via APIM proxy (subscription key)

APIM accepts an APIM subscription key and transparently injects a managed-identity Entra token when forwarding to the Foundry Toolbox. Clients no longer need to manage Foundry credentials.

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to track the policy execution.

In [ ]:
async def list_toolbox_tools_via_apim(endpoint, subscription_key):
    # APIM subscription key replaces the need for an Entra token
    headers = {"Ocp-Apim-Subscription-Key": subscription_key}
    try:
        async with httpx.AsyncClient(headers=headers) as mcp_http_client:
            async with streamable_http_client(endpoint, http_client=mcp_http_client) as streams:
                async with ClientSession(streams[0], streams[1]) as session:
                    await session.initialize()
                    response = await session.list_tools()
        print(f"✅ Connected via APIM proxy: {endpoint}")
        print("⚙️ Available tools (via APIM):")
        for tool in response.tools:
            print(f"   - {tool.name}: {tool.description}")
    except Exception as e:
        print(f"❌ Error: {e}")

# Use APIM native MCP endpoint
apim_mcp_test_url = f"{apim_resource_gateway_url}/toolbox/mcp-native"
print(f"Testing APIM MCP endpoint: {apim_mcp_test_url}")

# Test via the APIM native MCP endpoint (subscription key only — no Entra token needed)
asyncio.run(list_toolbox_tools_via_apim(apim_mcp_test_url, api_key))

<a id='8'></a>
### 🧪 Run chat completions with Toolbox tools via APIM (multiple examples)

Both the inference calls (to the model) and the tool calls (to the Foundry Toolbox) are routed through APIM. APIM provides centralized governance over the complete agent-tool interaction loop.

This section runs multiple prompts and prints a clear trace of each tool call (name, arguments, and result snippet).

In [ ]:
import json, asyncio
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from openai import AzureOpenAI
import nest_asyncio
nest_asyncio.apply()

async def call_tool(session, function_name, function_args):
    try:
        result = await session.call_tool(function_name, function_args)
        return str(result.content)
    except Exception as e:
        return json.dumps({"error": str(e)})

async def run_with_toolbox(prompt, scenario_name):
    """Run one chat completion with tools from the Foundry Toolbox, all routed through APIM."""
    headers = {"Ocp-Apim-Subscription-Key": api_key}
    print("\n" + "=" * 90)
    print(f"🧪 Scenario: {scenario_name}")
    print(f"📝 Prompt: {prompt}")
    print("=" * 90)

    try:
        # APIM native MCP server endpoint
        apim_mcp_tools_url = f"{apim_resource_gateway_url}/toolbox/mcp-native"
        async with httpx.AsyncClient(headers=headers) as mcp_http_client:
            async with streamable_http_client(apim_mcp_tools_url, http_client=mcp_http_client) as streams:
                async with ClientSession(streams[0], streams[1]) as session:
                    await session.initialize()

                    # Discover tools from the Toolbox
                    tools_response = await session.list_tools()
                    openai_tools = [{
                        "type": "function",
                        "function": {"name": tool.name, "parameters": tool.inputSchema}
                    } for tool in tools_response.tools]
                    print(f"✅ Discovered {len(openai_tools)} tool(s) from Toolbox via APIM")

                    # Route inference calls through APIM as well
                    client = AzureOpenAI(
                        azure_endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
                        api_key=api_key,
                        api_version=inference_api_version
                    )

                    # Step 1: let the model decide which tools to call
                    print("▶️ Step 1: model selects tools based on the prompt")
                    messages = [{"role": "user", "content": prompt}]
                    response = client.chat.completions.create(
                        model=models_config[0]['name'],
                        messages=messages,
                        tools=openai_tools
                    )
                    response_message = response.choices[0].message
                    tool_calls = response_message.tool_calls

                    if tool_calls:
                        print(f"🔧 Tool calls requested: {len(tool_calls)}")
                        messages.append(response_message)

                        # Step 2: call each tool via the APIM-proxied Toolbox endpoint
                        print("▶️ Step 2: calling tools via APIM -> Foundry Toolbox")
                        for i, tool_call in enumerate(tool_calls, start=1):
                            fn_name = tool_call.function.name
                            fn_args = json.loads(tool_call.function.arguments)
                            print(f"   [{i}] Tool: {fn_name}")
                            print(f"       Args: {json.dumps(fn_args, indent=2)}")

                            fn_result = await call_tool(session, fn_name, fn_args)
                            preview = fn_result[:240] + "..." if len(fn_result) > 240 else fn_result
                            print(f"       Result Preview: {preview}")

                            messages.append({
                                "tool_call_id": tool_call.id,
                                "role": "tool",
                                "name": fn_name,
                                "content": fn_result
                            })

                        # Step 3: final completion with tool results
                        print("▶️ Step 3: final answer from model")
                        final = client.chat.completions.create(
                            model=models_config[0]['name'],
                            messages=messages
                        )
                        print("💬 Final Answer:")
                        print(final.choices[0].message.content)
                    else:
                        print("ℹ️ No tool call was required for this prompt.")
                        print("💬", response_message.content)

    except Exception as e:
        print(f"❌ Error: {e}")


example_scenarios = [
    {
        "name": "Clinical Summary + Coverage Check",
        "prompt": "Buddy is a dog and the owner is Laura Smith. Policy number is PET-1042. Get Buddy's recent clinical summary and check coverage for a periodontal dental cleaning under this policy. Return both the clinical summary and coverage outcome in one response."
    },
    {
        "name": "Pre-Authorization Workflow",
        "prompt": "For pet Luna with policy PET-1042, request pre-authorization for orthopedic surgery estimated at 3200, then tell me next steps."
    },
    {
        "name": "Claims Follow-up",
        "prompt": "Check claim status for CLM-77821 and submit these supporting documents: invoice.pdf, referral-letter.pdf, discharge-summary.pdf."
    }
]

for scenario in example_scenarios:
    asyncio.run(run_with_toolbox(scenario["prompt"], scenario["name"]))

<a id='9'></a>
### 📋 Manage Toolbox versions

Toolbox versions are immutable snapshots. Create a new version, test it, then promote it to default — agents always connect to the default version via the APIM proxy, so promotion requires no APIM or agent changes.

In [ ]:
# List all versions of the Toolbox
versions = list(project.toolboxes.list_versions(name=toolbox_name))
utils.print_info(f"Toolbox '{toolbox_name}' has {len(versions)} version(s):")
for v in versions:
    tools_in_version = [t.get('server_label', t.get('type', '?')) for t in (v.tools or [])]
    print(f"   Version {v.version} — tools: {tools_in_version}")

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.